![Header Image](../assets/header_image.png "Header Image")

# Notebook 3 : Introduction à ROS 2 — Foxy Fitzroy

Bienvenue dans ce tutoriel d'introduction à **ROS 2** (Robot Operating System 2) !

ROS 2 est le système de communication standard utilisé dans les véhicules autonomes modernes,
les robots industriels, et les drones. Dans ce cours sur la **perception autonome**,
comprendre ROS 2 vous permettra d'interfacer des capteurs, de traiter des données en temps réel,
et de partager des résultats entre les modules de traitement.

> **Kernel requis :** Sélectionnez **"Python 3.8 (ROS 2 Foxy)"** en haut à droite de JupyterLab.
> (**Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**)

Dans ce notebook, vous allez

- **comprendre l'architecture DDS** de ROS 2 et ses différences fondamentales avec ROS 1
- **créer des nœuds ROS 2** selon l'approche orientée objet (classe `Node`) — la façon professionnelle
- **utiliser les Timers** pour des publications périodiques automatiques
- **pratiquer Publisher / Subscriber** avec des messages de conduite autonome
- **comprendre les QoS** (Quality of Service) pour adapter la fiabilité des communications
- **utiliser les Paramètres** pour configurer un nœud dynamiquement
- **inspecter le graphe ROS 2** depuis le terminal

---
# Partie 1 — Architecture et Concepts Fondamentaux

## 1.1 Pourquoi ROS 2 ?

ROS 1 a été créé en 2007 pour la recherche universitaire en robotique. Il est devenu le standard
de facto, mais souffre de limitations importantes pour les applications industrielles et les véhicules autonomes :

- **Point de défaillance unique** : tout le système s'arrête si `roscore` tombe
- **Pas de temps réel** : les systèmes de sécurité (freins, direction) exigent des garanties temporelles
- **Pas de sécurité** : n'importe quel nœud peut publier sur n'importe quel topic
- **Linux uniquement** : impossible d'intégrer des composants Windows ou embarqués

ROS 2 (2017) a été conçu depuis zéro pour corriger ces problèmes :

| Aspect | ROS 1 | ROS 2 |
|--------|--------|--------|
| **Middleware** | Architecture centralisée (`roscore`) | **DDS** — décentralisé, peer-to-peer |
| **Nœud maître** | `roscore` obligatoire | **Aucun master** |
| **Bibliothèque Python** | `rospy` | **`rclpy`** |
| **Temps réel** | Non supporté | Supporté (RCLCPP + DDS RT) |
| **Sécurité** | Absente | **DDS-Security** intégrée |
| **Plateformes** | Linux uniquement | Linux, Windows, macOS, RTOS |
| **QoS** | Aucun contrôle | **Profils QoS configurables** |
| **Actions** | Bibliothèque séparée | **Intégrées nativement** |
| **Cycle de vie** | Manuel | **Lifecycle Nodes** |
| **Ce conteneur** | Noetic (Ubuntu 20.04) | **Foxy (Ubuntu 20.04)** |

## 1.2 Le Middleware DDS — Comment ça marche ?

DDS (Data Distribution Service) est un standard industriel de communication publié par l'OMG.
C'est le même middleware utilisé dans les systèmes de contrôle d'avions de combat, de sous-marins,
et de véhicules autonomes en production.

### Découverte automatique
Quand un nœud ROS 2 démarre, DDS **diffuse sa présence** sur le réseau local. Les autres nœuds
le découvrent automatiquement — sans annuaire central.

```
ROS 1 (centralisé)                    ROS 2 (décentralisé — DDS)
═══════════════════                    ══════════════════════════

  Nœud A ──►  roscore  ◄── Nœud B       Nœud A ◄══════════► Nœud B
               (master)                     ║                   ║
                 ▲                          ╚══════► Nœud C ◄═══╝
               Nœud C                   (découverte DDS automatique)

  Si roscore tombe → tout s'arrête     Si Nœud B tombe → A et C continuent
```

### Domaines DDS
Les nœuds d'un même **domaine DDS** (`ROS_DOMAIN_ID`, défaut=0) se voient entre eux.
Changer le domaine isole des groupes de nœuds — utile en production pour séparer
les sous-systèmes (perception, planification, contrôle).

## 1.3 Les Primitives de Communication ROS 2

ROS 2 offre 4 modes de communication :

### Topics — Publication/Abonnement (asynchrone)
Un publisher envoie des messages en continu. N'importe quel subscriber peut écouter.
Cas d'usage : données capteurs (LiDAR → `/scan`), images caméra (→ `/camera/image_raw`).

### Services — Requête/Réponse (synchrone)
Un client envoie une requête et attend la réponse d'un serveur.
Cas d'usage : demander la classification d'une image, activer un mode de conduite.

### Actions — Tâches longues avec feedback
Comme les services, mais avec annulation et feedback périodique.
Cas d'usage : navigation vers un waypoint, manœuvre de stationnement.

### Paramètres — Configuration dynamique
Chaque nœud gère ses propres paramètres, modifiables à chaud sans redémarrage.
Cas d'usage : seuil de détection d'obstacle, vitesse maximale autorisée.

```
           TOPICS            SERVICES            ACTIONS
        (asynchrone)        (synchrone)     (longue durée)

  Pub ──────────► Sub      Client ⇄ Server   Client ⇄ Server
  (fire & forget)          (req / rep)       + feedback + annulation
```

---
# Partie 2 — Configuration de l'Environnement

In [ ]:
!source /opt/ros/foxy/setup.bash

### Vérification du kernel Python

ROS 2 Foxy est compilé pour **Python 3.8** (Python système Ubuntu 20.04).
Le kernel conda Python 3.9 ne peut pas charger les extensions C de `rclpy`.

> Si la cellule suivante affiche une erreur, changez de kernel :
> **Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**

In [ ]:
import sys
sys.path.insert(0, '/opt/ros/foxy/lib/python3.8/site-packages/')

import platform
ver = platform.python_version_tuple()
print(f"Python {platform.python_version()} — {'OK ✓' if ver[1]=='8' else 'ERREUR : attendu 3.8'}")
assert ver[1] == '8', "Sélectionnez 'Python 3.8 (ROS 2 Foxy)' dans Kernel >> Change Kernel."

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, HistoryPolicy

from std_msgs.msg import String, Float32
from geometry_msgs.msg import Twist, PoseStamped
from sensor_msgs.msg import LaserScan

import threading
import time
import math

print("rclpy", rclpy.__version__ if hasattr(rclpy, '__version__') else "importé")
print("Toutes les bibliothèques ROS 2 importées avec succès !")

---
# Partie 3 — Créer un Nœud ROS 2

## 3.1 Approche fonctionnelle (rapide, pour les scripts simples)

```python
rclpy.init()
node = rclpy.create_node('mon_noeud')
```

## 3.2 Approche orientée objet (recommandée pour les vrais projets)

La pratique standard en ROS 2 est de définir une **classe héritant de `Node`**.
Cela permet d'encapsuler l'état du nœud, de mieux structurer le code, et d'utiliser
les fonctionnalités avancées comme les **Lifecycle Nodes** et les **timers intégrés**.

```python
class MonNoeud(Node):
    def __init__(self):
        super().__init__('mon_noeud')   # nom du nœud
        # publishers, subscribers, timers créés ici
```

Nous allons créer un **nœud de statut de véhicule** qui publie périodiquement
l'état d'un véhicule autonome simulé.

In [ ]:
rclpy.init()

class NoeudVehicule(Node):
    """Nœud ROS 2 représentant un véhicule autonome simulé."""

    def __init__(self):
        super().__init__('vehicule_autonome')

        # Publisher : commandes de vitesse envoyées au contrôleur
        self.pub_cmd   = self.create_publisher(Twist,  '/cmd_vel',     10)
        # Publisher : vitesse courante (données odométrie)
        self.pub_vitesse = self.create_publisher(Float32, '/vitesse_kmh', 10)
        # Publisher : messages de statut lisibles
        self.pub_statut = self.create_publisher(String,  '/statut',     10)

        # Timer : toutes les 500 ms, appelle la fonction de mise à jour
        self.timer = self.create_timer(0.5, self._mise_a_jour)

        # État interne
        self._t0     = time.time()
        self._vitesse = 0.0   # m/s
        self._compteur = 0

        self.get_logger().info("Nœud vehicule_autonome démarré")

    def _mise_a_jour(self):
        """Callback appelé toutes les 500 ms par le timer."""
        t = time.time() - self._t0
        self._compteur += 1

        # Simuler une accélération progressive (0 → 50 km/h en 10 s)
        self._vitesse = min(50.0 / 3.6, (t / 10.0) * 50.0 / 3.6)

        # Publier la commande de vitesse
        cmd = Twist()
        cmd.linear.x = self._vitesse
        self.pub_cmd.publish(cmd)

        # Publier la vitesse en km/h
        msg_v = Float32()
        msg_v.data = float(self._vitesse * 3.6)
        self.pub_vitesse.publish(msg_v)

        # Publier le statut textuel
        statut = String()
        statut.data = f"t={t:.1f}s | v={self._vitesse*3.6:.1f} km/h | tick #{self._compteur}"
        self.pub_statut.publish(statut)

noeud_vehicule = NoeudVehicule()

# Lancer le spin dans un thread d'arrière-plan (rclpy.spin est bloquant)
spin_thread = threading.Thread(target=rclpy.spin, args=(noeud_vehicule,), daemon=True)
spin_thread.start()

print(f"Nœud '{noeud_vehicule.get_name()}' démarré avec timer à 2 Hz.")
print("Topics publiés : /cmd_vel | /vitesse_kmh | /statut")

## 3.3 Le Logger ROS 2

ROS 2 dispose d'un système de logs intégré avec 5 niveaux de sévérité :
`DEBUG` < `INFO` < `WARN` < `ERROR` < `FATAL`.

Le logger est accessible via `self.get_logger()` dans une classe Node, ou `node.get_logger()`.

In [ ]:
noeud_vehicule.get_logger().info("Démarrage nominal du véhicule")
noeud_vehicule.get_logger().warn("Vitesse > 30 km/h — zone scolaire détectée")
noeud_vehicule.get_logger().error("Obstacle à 2m — freinage d'urgence")

# Les messages de log apparaissent aussi dans le terminal avec source/timestamp

---
# Partie 4 — Timers : Publication Périodique Automatique

Les **timers** sont l'un des outils les plus importants en ROS 2.
Ils permettent d'exécuter une fonction à intervalles réguliers — essentiel pour :
- Publier des données capteurs à fréquence fixe (LIDAR à 10 Hz, caméra à 30 Hz)
- Exécuter une loi de commande à fréquence contrôlée (contrôleur PID à 100 Hz)
- Envoyer des messages de heartbeat (état du véhicule à 1 Hz)

**Syntaxe :**
```python
self.timer = self.create_timer(periode_secondes, fonction_callback)
```

Le timer ci-dessous observe le statut publié par le nœud véhicule pendant 3 secondes.

In [ ]:
# Observer les messages de statut publiés par le timer du nœud véhicule
messages_recus = []

def callback_statut(msg):
    messages_recus.append(msg.data)
    print(f"[/statut] {msg.data}")

sub_statut = noeud_vehicule.create_subscription(String, '/statut', callback_statut, 10)

print("Observation du topic /statut pendant 3 secondes...")
time.sleep(3.0)
print(f"\n{len(messages_recus)} messages reçus — fréquence réelle : {len(messages_recus)/3.0:.1f} Hz")

In [ ]:
# Observer les vitesses publiées par le timer
vitesses = []

def callback_vitesse(msg):
    vitesses.append(msg.data)

sub_vitesse = noeud_vehicule.create_subscription(Float32, '/vitesse_kmh', callback_vitesse, 10)

print("Collecte des vitesses pendant 5 secondes...")
time.sleep(5.0)

if vitesses:
    print(f"\n{len(vitesses)} échantillons collectés")
    print(f"Vitesse initiale : {vitesses[0]:.1f} km/h")
    print(f"Vitesse finale   : {vitesses[-1]:.1f} km/h")
    print(f"Vitesse max      : {max(vitesses):.1f} km/h")

In [ ]:
# Visualiser la courbe de vitesse collectée via le subscriber
import matplotlib.pyplot as plt
import numpy as np

if vitesses:
    t_ax = np.linspace(0, len(vitesses) * 0.5, len(vitesses))
    plt.figure(figsize=(9, 3))
    plt.plot(t_ax, vitesses, color='steelblue', linewidth=2)
    plt.axhline(50, color='red', linestyle='--', label='Limite 50 km/h')
    plt.xlabel('Temps (s)')
    plt.ylabel('Vitesse (km/h)')
    plt.title('Vitesse du véhicule reçue depuis /vitesse_kmh (ROS 2 Timer + Subscriber)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
# Partie 5 — Quality of Service (QoS)

C'est l'une des nouveautés les plus importantes de ROS 2. Les **profils QoS** permettent de configurer
exactement comment les messages sont transmis entre publishers et subscribers.

## Les 3 paramètres principaux

### Reliability (Fiabilité)
- `RELIABLE` : chaque message est garanti d'arriver (retransmission si perte). Idéal pour les commandes critiques.
- `BEST_EFFORT` : on fait au mieux, sans retransmission. Idéal pour les capteurs à haute fréquence (perdre quelques frames LIDAR est acceptable).

### Durability (Durabilité)
- `VOLATILE` : les messages sont perdus si personne n'écoute au moment de la publication.
- `TRANSIENT_LOCAL` : le dernier message est conservé et envoyé aux nouveaux subscribers qui arrivent après. Idéal pour le `robot_description` ou les cartes.

### History (Historique)
- `KEEP_LAST(N)` : garder les N derniers messages dans la file d'attente.
- `KEEP_ALL` : garder tous les messages.

## Profils prédéfinis

| Profil | Usage typique |
|--------|---------------|
| `qos_profile_sensor_data` | Capteurs temps réel (LiDAR, caméra) |
| `qos_profile_parameters` | Paramètres des nœuds |
| `qos_profile_services_default` | Services |
| `QoSProfile(depth=10)` | Topic généraliste (défaut) |

In [ ]:
from rclpy.qos import qos_profile_sensor_data

# QoS RELIABLE — pour les commandes critiques (ne pas perdre de message)
qos_commandes = QoSProfile(
    reliability = ReliabilityPolicy.RELIABLE,
    durability  = DurabilityPolicy.VOLATILE,
    history     = HistoryPolicy.KEEP_LAST,
    depth       = 10
)

# QoS BEST_EFFORT — pour les capteurs haute fréquence (perdre des frames est OK)
qos_capteurs = QoSProfile(
    reliability = ReliabilityPolicy.BEST_EFFORT,
    durability  = DurabilityPolicy.VOLATILE,
    history     = HistoryPolicy.KEEP_LAST,
    depth       = 5
)

# QoS TRANSIENT_LOCAL — pour la configuration (nouveaux subscribers reçoivent le dernier message)
qos_config = QoSProfile(
    reliability = ReliabilityPolicy.RELIABLE,
    durability  = DurabilityPolicy.TRANSIENT_LOCAL,
    history     = HistoryPolicy.KEEP_LAST,
    depth       = 1
)

# Créer des publishers avec différents profils QoS
pub_commande_fiable = noeud_vehicule.create_publisher(Twist,  '/cmd_vel_safe', qos_commandes)
pub_scan_rapide     = noeud_vehicule.create_publisher(LaserScan, '/scan_raw',  qos_capteurs)
pub_config_map      = noeud_vehicule.create_publisher(String, '/carte_statut', qos_config)

# Publier un message de configuration (TRANSIENT_LOCAL — les futurs subscribers le recevront)
msg_config = String()
msg_config.data = "Carte chargée : parking_campus_v3.pgm (256x256, 0.05 m/px)"
pub_config_map.publish(msg_config)

print("QoS configurés :")
print("  /cmd_vel_safe  → RELIABLE   (commandes critiques)")
print("  /scan_raw      → BEST_EFFORT (données LIDAR haute fréquence)")
print("  /carte_statut  → TRANSIENT_LOCAL (nouveaux subscribers reçoivent la carte)")

### Compatibilité QoS Publisher ↔ Subscriber

Un subscriber ne peut recevoir les messages d'un publisher que si leurs profils QoS sont **compatibles** :

| Publisher | Subscriber | Résultat |
|-----------|------------|----------|
| RELIABLE | RELIABLE | Compatible ✓ |
| RELIABLE | BEST_EFFORT | Compatible ✓ |
| BEST_EFFORT | RELIABLE | **Incompatible ✗** (le subscriber exige plus que ce que le publisher offre) |
| BEST_EFFORT | BEST_EFFORT | Compatible ✓ |

> **Astuce pratique :** Si vous ne recevez aucun message d'un topic, vérifiez la compatibilité
> QoS avec `ros2 topic info /nom_topic --verbose`.

---
# Partie 6 — Paramètres de Nœud

Les **paramètres** permettent de configurer un nœud sans modifier son code ni le redémarrer.
En ROS 2, chaque nœud gère ses propres paramètres (contrairement au serveur de paramètres global de ROS 1).

**Workflow typique :**
1. Déclarer le paramètre avec sa valeur par défaut
2. Lire sa valeur dans le code
3. Modifier depuis le terminal ou un fichier YAML en production

```bash
# Modifier un paramètre depuis le terminal
ros2 param set /vehicule_autonome vitesse_max 30.0

# Voir tous les paramètres d'un nœud
ros2 param list /vehicule_autonome

# Lire la valeur d'un paramètre
ros2 param get /vehicule_autonome vitesse_max
```

In [ ]:
# Déclarer des paramètres sur le nœud véhicule
noeud_vehicule.declare_parameter('vitesse_max',      50.0)   # km/h
noeud_vehicule.declare_parameter('distance_securite', 2.5)   # mètres
noeud_vehicule.declare_parameter('mode_conduite',   'normal')  # 'normal' | 'economique' | 'sport'

# Lire les valeurs des paramètres
v_max    = noeud_vehicule.get_parameter('vitesse_max').value
d_sec    = noeud_vehicule.get_parameter('distance_securite').value
mode     = noeud_vehicule.get_parameter('mode_conduite').value

print("Paramètres du nœud vehicule_autonome :")
print(f"  vitesse_max       = {v_max} km/h")
print(f"  distance_securite = {d_sec} m")
print(f"  mode_conduite     = '{mode}'")

In [ ]:
# Modifier un paramètre depuis Python (équivalent de ros2 param set)
from rclpy.parameter import Parameter

noeud_vehicule.set_parameters([
    Parameter('vitesse_max',    Parameter.Type.DOUBLE, 30.0),
    Parameter('mode_conduite',  Parameter.Type.STRING, 'economique'),
])

v_max_new = noeud_vehicule.get_parameter('vitesse_max').value
mode_new  = noeud_vehicule.get_parameter('mode_conduite').value

print(f"Paramètres mis à jour :")
print(f"  vitesse_max   : 50.0 → {v_max_new} km/h")
print(f"  mode_conduite : 'normal' → '{mode_new}'")

---
# Partie 7 — Publisher / Subscriber : Cas d'Usage Conduite Autonome

## 7.1 Le message PoseStamped

`PoseStamped` est la position et l'orientation du véhicule avec un timestamp.
C'est l'un des messages les plus utilisés en conduite autonome :
position GPS/odométrie → module de planification → `/goal_pose`.

```
std_msgs/Header header
    builtin_interfaces/Time stamp
    string frame_id          ← référentiel ('map', 'odom', 'base_link'...)
geometry_msgs/Pose pose
    Point position           ← x, y, z (mètres)
    Quaternion orientation   ← x, y, z, w
```

In [ ]:
# Publisher : position actuelle du véhicule (odométrie)
pub_pose = noeud_vehicule.create_publisher(PoseStamped, '/pose_vehicule', 10)

# Subscriber : waypoints de destination reçus du module de planification
waypoints_recus = []

def callback_waypoint(msg):
    waypoints_recus.append((msg.pose.position.x, msg.pose.position.y))
    print(f"[/waypoint] Destination : ({msg.pose.position.x:.1f}, {msg.pose.position.y:.1f}) "
          f"dans le repère '{msg.header.frame_id}'")

sub_waypoint = noeud_vehicule.create_subscription(PoseStamped, '/waypoint', callback_waypoint, 10)

# Publier la position actuelle
def publier_position(x, y, theta, frame='odom'):
    msg = PoseStamped()
    msg.header.frame_id = frame
    msg.header.stamp    = noeud_vehicule.get_clock().now().to_msg()
    msg.pose.position.x = x
    msg.pose.position.y = y
    msg.pose.position.z = 0.0
    # Quaternion pour l'angle theta (rotation autour de Z)
    msg.pose.orientation.z = math.sin(theta / 2)
    msg.pose.orientation.w = math.cos(theta / 2)
    pub_pose.publish(msg)
    print(f"[/pose_vehicule] Position publiée : ({x:.1f}, {y:.1f}) θ={math.degrees(theta):.0f}°")

# Simuler le véhicule à différentes positions sur un trajet
positions = [(0.0, 0.0, 0.0), (10.0, 2.0, 0.2), (25.0, 5.0, 0.1), (40.0, 3.0, -0.1)]
for x, y, theta in positions:
    publier_position(x, y, theta)
    time.sleep(0.1)

In [ ]:
# Publier un waypoint de destination (simulant le module de planification)
pub_waypoint = noeud_vehicule.create_publisher(PoseStamped, '/waypoint', 10)

waypoints = [(50.0, 0.0), (80.0, -5.0), (100.0, 0.0)]
for wx, wy in waypoints:
    msg = PoseStamped()
    msg.header.frame_id     = 'map'
    msg.header.stamp        = noeud_vehicule.get_clock().now().to_msg()
    msg.pose.position.x     = wx
    msg.pose.position.y     = wy
    msg.pose.orientation.w  = 1.0
    pub_waypoint.publish(msg)
    time.sleep(0.1)

time.sleep(0.3)  # Attendre les callbacks
print(f"\n{len(waypoints_recus)} waypoints reçus par le subscriber.")

## 7.2 Graphe de communication ROS 2

Voici le graphe des nœuds et topics créés dans ce notebook :

```
                    ┌───────────────────────────────┐
                    │      vehicule_autonome        │
                    │      (NoeudVehicule)           │
                    │                               │
                    │  Timer 2Hz ──► _mise_a_jour() │
                    └─────┬──────┬──────┬───────────┘
                          │      │      │
               /cmd_vel   │      │      │  /statut
                          │   /vitesse_kmh
                          ▼      ▼      ▼
                    [Subscribers dans ce notebook]
                    callback_statut | callback_vitesse

    [pub_waypoint] ──► /waypoint ──► callback_waypoint
    [pub_pose]     ──► /pose_vehicule
    [pub_config]   ──► /carte_statut  (TRANSIENT_LOCAL)
```

---
# Partie 8 — Inspecter le Graphe depuis le Terminal

Pendant que le nœud est actif, ouvrez un terminal JupyterLab
(**Fichier >> Nouveau >> Terminal**) et explorez :

```bash
source /opt/ros/foxy/setup.bash

# ── NŒUDS ──────────────────────────────────────────
# Lister les nœuds actifs
ros2 node list
# Afficher les détails d'un nœud (publishers, subscribers, services)
ros2 node info /vehicule_autonome

# ── TOPICS ─────────────────────────────────────────
# Lister tous les topics
ros2 topic list
# Voir les messages en temps réel
ros2 topic echo /statut
ros2 topic echo /vitesse_kmh
# Fréquence de publication mesurée
ros2 topic hz /cmd_vel
# Bande passante
ros2 topic bw /cmd_vel
# Informations + QoS du topic
ros2 topic info /cmd_vel_safe --verbose

# ── PARAMÈTRES ─────────────────────────────────────
# Lister les paramètres du nœud
ros2 param list /vehicule_autonome
# Lire un paramètre
ros2 param get /vehicule_autonome vitesse_max
# Modifier un paramètre à chaud
ros2 param set /vehicule_autonome vitesse_max 30.0

# ── TYPES DE MESSAGES ──────────────────────────────
# Voir la structure d'un type de message
ros2 interface show geometry_msgs/msg/Twist
ros2 interface show geometry_msgs/msg/PoseStamped
```

> **Rappel :** Tout fonctionne sans `roscore`. Essayez ces commandes immédiatement !

---
# Partie 9 — Exercice Pratique

Créez un nœud ROS 2 (classe héritant de `Node`) qui simule un **détecteur d'obstacles**.

**Spécifications :**
1. Le nœud s'appelle `'detecteur_obstacles'`
2. Il publie sur `/distance_obstacle` (`Float32`) à **4 Hz** (via un timer)
3. La distance simulée oscille entre 0.5 m et 8.0 m : `4.25 + 3.75 * sin(2π * 0.2 * t)`
4. Il publie sur `/alerte` (`String`) :
   - `"DANGER"` si distance < 2.0 m
   - `"ATTENTION"` si 2.0 ≤ distance < 5.0 m
   - `"OK"` si distance ≥ 5.0 m
5. Créez un subscriber dans le notebook qui affiche les alertes reçues

In [ ]:
# Complétez ce code

class DetecteurObstacles(Node):
    def __init__(self):
        super().__init__('detecteur_obstacles')
        self._t0 = time.time()

        # TODO : créer les publishers /distance_obstacle et /alerte
        # TODO : créer le timer à 4 Hz

    def _callback_timer(self):
        t = time.time() - self._t0
        distance = 4.25 + 3.75 * math.sin(2 * math.pi * 0.2 * t)

        # TODO : publier la distance
        # TODO : calculer l'alerte et la publier
        pass


# TODO : instancier le nœud, lancer le spin, créer le subscriber d'alertes
# Observez pendant 6 secondes puis affichez le nombre d'alertes DANGER reçues

print("À vous de compléter ce code !")

In [ ]:
# Solution de référence (ne regardez qu'après avoir essayé !)

class DetecteurObstaclesSolution(Node):
    def __init__(self):
        super().__init__('detecteur_obstacles')
        self._t0 = time.time()

        self.pub_dist   = self.create_publisher(Float32, '/distance_obstacle', 10)
        self.pub_alerte = self.create_publisher(String,  '/alerte',            10)
        self.timer      = self.create_timer(1.0 / 4.0, self._callback_timer)  # 4 Hz

    def _callback_timer(self):
        t        = time.time() - self._t0
        distance = 4.25 + 3.75 * math.sin(2 * math.pi * 0.2 * t)

        msg_d = Float32()
        msg_d.data = float(distance)
        self.pub_dist.publish(msg_d)

        if distance < 2.0:
            niveau = 'DANGER'
        elif distance < 5.0:
            niveau = 'ATTENTION'
        else:
            niveau = 'OK'

        msg_a = String()
        msg_a.data = f"{niveau} — {distance:.2f} m"
        self.pub_alerte.publish(msg_a)


detecteur = DetecteurObstaclesSolution()
spin_detecteur = threading.Thread(target=rclpy.spin, args=(detecteur,), daemon=True)
spin_detecteur.start()

alertes = []
def callback_alerte(msg):
    alertes.append(msg.data)
    print(f"[/alerte] {msg.data}")

sub_alerte = detecteur.create_subscription(String, '/alerte', callback_alerte, 10)

print("Détecteur démarré — observation 6 secondes...")
time.sleep(6.0)

n_danger   = sum(1 for a in alertes if 'DANGER'    in a)
n_attn     = sum(1 for a in alertes if 'ATTENTION' in a)
n_ok       = sum(1 for a in alertes if 'OK'        in a)
print(f"\nRésumé : {n_danger} DANGER | {n_attn} ATTENTION | {n_ok} OK ({len(alertes)} total)")

---
# Partie 10 — Arrêt Propre des Nœuds

Il est important de détruire les nœuds et d'appeler `rclpy.shutdown()` pour libérer
les ressources DDS proprement. Sans cela, le prochain `rclpy.init()` peut échouer.

In [ ]:
# Arrêter tous les nœuds créés dans ce notebook
for noeud in [noeud_vehicule, detecteur]:
    try:
        noeud.destroy_node()
        print(f"Nœud '{noeud.get_name()}' arrêté.")
    except Exception:
        pass

rclpy.shutdown()
print("rclpy shutdown — ressources DDS libérées.")

---
# Résumé

| Concept appris | Commande / Méthode clé |
|----------------|------------------------|
| Architecture DDS | Pas de roscore — découverte automatique |
| Nœud (OOP) | `class MonNoeud(Node)` + `super().__init__('nom')` |
| Timer | `self.create_timer(periode, callback)` |
| Publisher | `self.create_publisher(Type, '/topic', qos)` |
| Subscriber | `self.create_subscription(Type, '/topic', callback, qos)` |
| Spin (Jupyter) | `threading.Thread(target=rclpy.spin, ...)` |
| QoS Reliable | `ReliabilityPolicy.RELIABLE` — commandes critiques |
| QoS Best Effort | `ReliabilityPolicy.BEST_EFFORT` — capteurs haute fréquence |
| QoS Transient Local | `DurabilityPolicy.TRANSIENT_LOCAL` — configuration |
| Paramètres | `declare_parameter` + `get_parameter` + `set_parameters` |
| Logger | `self.get_logger().info/warn/error()` |
| Inspection terminal | `ros2 node info`, `ros2 topic echo/hz/bw`, `ros2 param list` |

**Prochaine étape :** Le notebook `4_message_visualization_ros2_fr.ipynb` vous montrera comment
visualiser ces données en temps réel avec matplotlib et ipywidgets.